In [1]:
!pip install -U transformers accelerate bitsandbytes librosa sentencepiece av audioread decord torchvision requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 73.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 52.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 92.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 101.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 3.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 4.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 10.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 9.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 30.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import shutil

# Dọn dẹp để tránh xung đột cache
if os.path.exists("qwen_omni_utils"):
    shutil.rmtree("qwen_omni_utils")

os.makedirs("qwen_omni_utils/v2_5", exist_ok=True)
# Phải có file __init__.py ở mọi cấp độ thư mục để Python nhận diện là package
with open("qwen_omni_utils/__init__.py", "w") as f: f.write("") 

print("✅ Đã làm sạch và tạo cấu trúc thư mục mới.")

✅ Đã làm sạch và tạo cấu trúc thư mục mới.


In [3]:
%%writefile qwen_omni_utils/v2_5/audio_process.py
import base64
from io import BytesIO
import audioread
import av
import librosa
import numpy as np

# Qwen3-Omni khuyến nghị dùng 24k để giữ độ chi tiết của âm thanh
SAMPLE_RATE = 24000

def _check_if_video_has_audio(video_path):
    try:
        container = av.open(video_path)
        audio_streams = [stream for stream in container.streams if stream.type == "audio"]
        return len(audio_streams) > 0
    except:
        return False

def process_audio_info(conversations, use_audio_in_video=False):
    audios = []
    # Xử lý cả dạng list đơn hoặc list của list
    if isinstance(conversations[0], dict):
        conversations = [conversations]
        
    for conversation in conversations:
        for message in conversation:
            content = message.get("content")
            if not isinstance(content, list):
                continue
                
            for ele in content:
                data = None
                audio_start = ele.get("audio_start", ele.get("video_start", 0.0))
                audio_end = ele.get("audio_end", ele.get("video_end", None))
                
                # Case 1: Audio trực tiếp
                if ele["type"] == "audio":
                    path = ele.get("audio", ele.get("audio_url"))
                    if isinstance(path, np.ndarray):
                        if path.ndim > 1: raise ValueError("Support only mono audio")
                        start_idx = int(SAMPLE_RATE * audio_start)
                        end_idx = None if audio_end is None else int(SAMPLE_RATE * audio_end)
                        audios.append(path[start_idx:end_idx])
                        continue
                    elif path.startswith("data:audio"):
                        _, base64_data = path.split("base64,", 1)
                        data = BytesIO(base64.b64decode(base64_data))
                    elif path.startswith(("http://", "https://")):
                        data = audioread.ffdec.FFmpegAudioFile(path)
                    else:
                        data = path.replace("file://", "")
                
                # Case 2: Trích xuất từ Video
                elif use_audio_in_video and ele["type"] == "video":
                    path = ele.get("video", ele.get("video_url")).replace("file://", "")
                    if _check_if_video_has_audio(path):
                        data = path
                    else:
                        continue # Bỏ qua nếu video không có tiếng
                
                else:
                    continue

                # Load và Resample bằng librosa
                if data:
                    duration = (audio_end - audio_start) if audio_end is not None else None
                    y, _ = librosa.load(data, sr=SAMPLE_RATE, offset=audio_start, duration=duration)
                    audios.append(y)
                    
    return audios if len(audios) > 0 else None

Writing qwen_omni_utils/v2_5/audio_process.py


In [4]:
%%writefile qwen_omni_utils/v2_5/vision_process.py
from __future__ import annotations
import base64
import copy
import logging
import math
import os
import sys
import time
import warnings
from functools import lru_cache
from io import BytesIO
from typing import Optional

import requests
import torch
import torchvision
from packaging import version
from PIL import Image
from torchvision import io, transforms
from torchvision.transforms import InterpolationMode

logger = logging.getLogger(__name__)

# Constants cho Qwen3-Omni
IMAGE_FACTOR = 28
MIN_PIXELS = 4 * 28 * 28
MAX_PIXELS = 16384 * 28 * 28
MAX_RATIO = 200
VIDEO_MIN_PIXELS = 128 * 28 * 28
VIDEO_MAX_PIXELS = 768 * 28 * 28
FRAME_FACTOR = 2
FPS = 2.0
FPS_MIN_FRAMES = 4
FPS_MAX_FRAMES = 768

VIDEO_TOTAL_PIXELS = int(float(os.environ.get('VIDEO_MAX_PIXELS', 128000 * 28 * 28 * 0.9)))

def round_by_factor(number: int, factor: int) -> int:
    return round(number / factor) * factor

def ceil_by_factor(number: int, factor: int) -> int:
    return math.ceil(number / factor) * factor

def floor_by_factor(number: int, factor: int) -> int:
    return math.floor(number / factor) * factor

def smart_resize(height: int, width: int, factor: int = IMAGE_FACTOR, min_pixels: int = MIN_PIXELS, max_pixels: int = MAX_PIXELS) -> tuple[int, int]:
    if max(height, width) / min(height, width) > MAX_RATIO:
        raise ValueError(f"Aspect ratio too large: {max(height, width) / min(height, width)}")
    h_bar = max(factor, round_by_factor(height, factor))
    w_bar = max(factor, round_by_factor(width, factor))
    if h_bar * w_bar > max_pixels:
        beta = math.sqrt((height * width) / max_pixels)
        h_bar = max(factor, floor_by_factor(height / beta, factor))
        w_bar = max(factor, floor_by_factor(width / beta, factor))
    elif h_bar * w_bar < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h_bar = ceil_by_factor(height * beta, factor)
        w_bar = ceil_by_factor(width * beta, factor)
    return h_bar, w_bar

def to_rgb(pil_image: Image.Image) -> Image.Image:
    if pil_image.mode == "RGBA":
        white_background = Image.new("RGB", pil_image.size, (255, 255, 255))
        white_background.paste(pil_image, mask=pil_image.split()[3])
        return white_background
    return pil_image.convert("RGB")

def fetch_image(ele: dict, size_factor: int = IMAGE_FACTOR) -> Image.Image:
    image = ele.get("image", ele.get("image_url"))
    if isinstance(image, Image.Image):
        image_obj = image
    elif image.startswith(("http://", "https://")):
        resp = requests.get(image, stream=True)
        image_obj = Image.open(BytesIO(resp.content))
    elif image.startswith("file://"):
        image_obj = Image.open(image[7:])
    elif image.startswith("data:image"):
        _, base64_data = image.split("base64,", 1)
        image_obj = Image.open(BytesIO(base64.b64decode(base64_data)))
    else:
        image_obj = Image.open(image)
    
    image = to_rgb(image_obj)
    width, height = image.size
    resized_h, resized_w = smart_resize(height, width, factor=size_factor)
    return image.resize((resized_w, resized_h))

def smart_nframes(ele: dict, total_frames: int, video_fps: float) -> int:
    fps = ele.get("fps", FPS)
    nframes = total_frames / video_fps * fps
    nframes = min(min(max(nframes, FPS_MIN_FRAMES), FPS_MAX_FRAMES), total_frames)
    return floor_by_factor(int(nframes), FRAME_FACTOR)

def _read_video_torchvision(ele: dict) -> tuple[torch.Tensor, float]:
    video_path = ele["video"].replace("file://", "")
    video, _, info = io.read_video(video_path, pts_unit="sec", output_format="TCHW")
    total_frames, video_fps = video.size(0), info["video_fps"]
    nframes = smart_nframes(ele, total_frames, video_fps)
    idx = torch.linspace(0, total_frames - 1, nframes).round().long()
    return video[idx], (nframes / max(total_frames, 1e-6) * video_fps)

def fetch_video(ele: dict, image_factor: int = IMAGE_FACTOR, return_video_sample_fps: bool = False):
    # Dùng torchvision làm mặc định cho ổn định trên Kaggle
    video, sample_fps = _read_video_torchvision(ele)
    nframes, _, height, width = video.shape
    resized_h, resized_w = smart_resize(height, width, factor=image_factor, min_pixels=VIDEO_MIN_PIXELS, max_pixels=VIDEO_MAX_PIXELS)
    video = transforms.functional.resize(video, [resized_h, resized_w], interpolation=InterpolationMode.BICUBIC, antialias=True).float()
    return (video, sample_fps) if return_video_sample_fps else video

def extract_vision_info(conversations: list) -> list[dict]:
    vision_infos = []
    for msg in (conversations if isinstance(conversations[0], list) else [conversations]):
        for m in msg:
            if isinstance(m.get("content"), list):
                for ele in m["content"]:
                    if any(k in ele for k in ["image", "image_url", "video"]):
                        vision_infos.append(ele)
    return vision_infos

def process_vision_info(conversations: list, return_video_kwargs: bool = False):
    vision_infos = extract_vision_info(conversations)
    image_inputs, video_inputs, fps_list = [], [], []
    for info in vision_infos:
        if "image" in info or "image_url" in info:
            image_inputs.append(fetch_image(info))
        elif "video" in info:
            v_input, v_fps = fetch_video(info, return_video_sample_fps=True)
            video_inputs.append(v_input)
            fps_list.append(v_fps)
    
    res_images = image_inputs if image_inputs else None
    res_videos = video_inputs if video_inputs else None
    if return_video_kwargs:
        return res_images, res_videos, {'fps': fps_list}
    return res_images, res_videos

Writing qwen_omni_utils/v2_5/vision_process.py


In [5]:
%%writefile qwen_omni_utils/v2_5/__init__.py
from .audio_process import process_audio_info
from .vision_process import (
    extract_vision_info,
    fetch_image,
    fetch_video,
    process_vision_info,
    smart_resize,
)

def process_mm_info(conversations, use_audio_in_video=False, return_video_kwargs=False):
    # 1. Gọi logic xử lý âm thanh
    audios = process_audio_info(conversations, use_audio_in_video)
    
    # 2. Gọi logic xử lý hình ảnh/video từ vision_process
    vision_res = process_vision_info(conversations, return_video_kwargs=return_video_kwargs)
    
    # vision_res sẽ là (images, videos) hoặc (images, videos, kwargs) tùy tham số
    # Trả về bộ tuple (audios, images, videos, ...)
    return (audios,) + vision_res

Writing qwen_omni_utils/v2_5/__init__.py


In [6]:
import torch
import sys
# Đảm bảo Python nhận diện được folder local
if "." not in sys.path: sys.path.append(".")

from qwen_omni_utils.v2_5 import process_mm_info
from transformers import Qwen3OmniMoeProcessor

# Đường dẫn model trên Kaggle
MODEL_PATH = "/kaggle/input/models/khanhlong/qwen3-omni/pytorch/default/9"
processor = Qwen3OmniMoeProcessor.from_pretrained(MODEL_PATH)

# Định nghĩa công cụ và tin nhắn
tools_xml = """
<tools>
{
    'type': 'function', 
    'function': {
        'name': 'escalate_claim_discrepancy', 
        'description': 'Escalate insurance claim to manager.', 
        'parameters': {
            'type': 'object', 
            'properties': {
                'claim_id': {'type': 'string'},
                'issue_type': {'type': 'string', 'enum': ['payout_discrepancy', 'repair_vs_replacement']},
                'request_manager': {'type': 'boolean'}
            }, 
            'required': ['claim_id', 'issue_type', 'request_manager']
        }
    }
}
</tools>
"""

messages = [
    {"role": "system", "content": f"You are a helpful assistant. {tools_xml}\nReturn tool calls in <tool_call> tags."},
    {"role": "user", "content": [{"type": "audio", "audio": "/kaggle/input/datasets/khanhlong/audio-test/test.mp3"}]}
]

# 1. Áp dụng template chat
text_prompt = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

# 2. Tiền xử lý dữ liệu đa phương thức (gọi từ thư viện vừa tạo)
# Trình tự trả về: (audios, images, videos)
audios, images, videos = process_mm_info(messages, use_audio_in_video=False)

# 3. Tạo Inputs cho mô hình
inputs = processor(text=text_prompt, audio=audios, images=images, videos=videos, return_tensors="pt")

# 4. Lưu lại để dùng cho Cell chạy GPU
torch.save(inputs, "preprocessed_inputs.pt")
print(f"✅ Thành công! Đã tạo file preprocessed_inputs.pt")
print(f"Keys gửi vào LLM: {inputs.keys()}")

Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_section', 'interleaved'}


✅ Thành công! Đã tạo file preprocessed_inputs.pt
Keys gửi vào LLM: KeysView({'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
            715,     27,  15918,    397,    515,    262,    364,   1313,   1210,
            364,   1688,    516,    715,    262,    364,   1688,   1210,    341,
            286,    364,    606,   1210,    364,  81082,    349,  84969,   9932,
            837,    848,  11130,    516,    715,    286,    364,   4684,   1210,
            364,  36121,    278,    349,   8113,   3717,    311,   6645,  15670,
            715,    286,    364,  13786,   1210,    341,    310,    364,   1313,
           1210,    364,   1700,    516,    715,    310,    364,  13193,   1210,
            341,    394,    364,   7859,    842,   1210,   5360,   1313,   1210,
            364,    917,  11688,    394,    364,  11159,   1819,   1210,   5360,
           1313,   1210,    364,    917,    516,    364,   9018,   1210,   2509,
             79,   

In [10]:
# Cài đặt bản bitsandbytes tương thích tốt hơn với môi trường Kaggle 2026
!pip install -U bitsandbytes accelerate transformers

import os
import torch

# Fix lỗi thiếu libnvJitLink
cuda_path = "/usr/local/cuda/lib64"
if os.path.exists(cuda_path):
    os.environ['LD_LIBRARY_PATH'] = os.environ.get('LD_LIBRARY_PATH', '') + f":{cuda_path}"

# Kiểm tra xem bitsandbytes đã nhận được GPU chưa
import bitsandbytes as bnb
print(f"✅ Bitsandbytes CUDA version: {bnb.utils.get_cuda_lib_handle()}")

AttributeError: module 'bitsandbytes.utils' has no attribute 'get_cuda_lib_handle'

In [1]:
from transformers import Qwen3OmniMoeForConditionalGeneration, BitsAndBytesConfig, Qwen3OmniMoeProcessor
import shutil

MODEL_PATH = "/kaggle/input/models/khanhlong/qwen3-omni/pytorch/default/9"
OFFLOAD_DIR = "/kaggle/working/offload_folder"

# Dọn dẹp folder offload cũ nếu có
if os.path.exists(OFFLOAD_DIR):
    shutil.rmtree(OFFLOAD_DIR)
os.makedirs(OFFLOAD_DIR, exist_ok=True)

# 1. Cấu hình 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True # Cho phép đẩy các layer MoE nặng sang CPU/Disk
)

# 2. Tinh chỉnh bộ nhớ (MoE cần nhiều RAM hệ thống hơn để điều phối)
# T4 có 16GB, ta dùng 14.5GB mỗi card
max_memory = {0: "14.5GiB", 1: "14.5GiB", "cpu": "24GiB"}

# 3. Load Model
model = Qwen3OmniMoeForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory,
    offload_folder=OFFLOAD_DIR, # Giải quyết lỗi ValueError của bạn
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

print(f"🚀 Model loaded! Device map: {model.hf_device_map}")

NameError: name 'os' is not defined

In [7]:
import torch
from transformers import Qwen3OmniMoeForConditionalGeneration, BitsAndBytesConfig, Qwen3OmniMoeProcessor

MODEL_PATH = "/kaggle/input/models/khanhlong/qwen3-omni/pytorch/default/9"

# 1. Tải lại Processor (Bắt buộc để giải mã kết quả)
processor = Qwen3OmniMoeProcessor.from_pretrained(MODEL_PATH)

# 2. Cấu hình Quantization tối ưu cho T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, # Giúp tính toán nhanh và chính xác hơn trên T4
    bnb_4bit_use_double_quant=True
)

# 3. Load Model với cơ chế cân bằng GPU
model = Qwen3OmniMoeForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto", 
    trust_remote_code=True,
    low_cpu_mem_usage=True # Tránh tràn RAM hệ thống của Kaggle khi load model 30B
)

# 4. Nạp và đẩy dữ liệu lên đúng thiết bị
inputs = torch.load("preprocessed_inputs.pt")

# Đưa tất cả tensor về đúng device mà model đang đứng đầu (thường là cuda:0)
# và ép kiểu về bfloat16 để đồng bộ với model
inputs = {k: v.to(model.device).to(torch.bfloat16) if v.dtype == torch.float32 else v.to(model.device) 
          for k, v in inputs.items()}

# 5. Sinh mã (Inference)
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False, 
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id
    )

# 6. Giải mã (Chỉ lấy phần text mới được sinh ra)
input_len = inputs['input_ids'].shape[1]
response_ids = output_ids[:, input_len:] # Bỏ qua phần prompt cũ
response = processor.batch_decode(response_ids, skip_special_tokens=True)[0]

print("-" * 30)
print("MODEL RESPONSE (TOOL DIRECTION):")
print(response.strip())
print("-" * 30)

Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_section', 'interleaved'}
Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_section', 'interleaved'}
bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 